# Estudio y comparativa resultados distributed + neg out y high fixed

En este notebook vamos a analizar los resultados de la versión distribuida basada en complejidad que incluye la opción de ir quitando las variables con complejidad negativa (con un margen de tolerancia) y de ir fijando las que tienen un valor alto de complejidad. Este valor alto de complejidad es difícil de establecer porque siempre son valores bajitos y además varían en función de la medida de complejidad. Así, por ahora (y por temas computacionales) solo estamos ejecutando para kDN y fijándonos en sus valores. Además, debido a que es "más seguro" que las negativas son malas, damos más fuerza a esto. Es decir, si una variable toma valores negativos, se saca aunque previamente se hubiera fijado. Una vez tomas algún valor negativo estás totalmente fuera y no puedes volver a entrar. Pero si se fija la variable, sí puede volver a salir. Digamos que sacamos variables con más fuerza para ir limpiando.

En base a los resultados del notebook "Distributed_FS_Complexity_Analysis", ya solo aplicamos un método a la hora de obtener los resultados. Se ejecuta únicamente la opción "random choice". Luego el método es: tomar muestras bootstrap de variables e ir evaluando la complejidad siguiendo un esquema backward. La variable que sale en cada caso se escoge de manera aleatoria.

Aquí vamos tanto a estudiar los resultados de esta versión distribuida como a comparar su performance tanto en complejidad como en rendimiento con los métodos del SOTA. En todos los casos vamos a escoger el número k de variables como el número de variables informativas (sabemos cuál es porque estamos en el caso artificial). En nuestra versión distribuida hemos hecho un filtro previo de variables con correlación de Pearson superior a 0.9. Así, para que las comparaciones sean justas, hemos ejecutado los métodos del estado del arte tanto sin el filtro como con el filtro. Los métodos del SOTA con filtro tendrán un "_corr" en el nombre.

In [ ]:
import copy
from sklearn import preprocessing
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.feature_selection import mutual_info_classif, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from skrebate import ReliefF
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
import glob
import re
from DistributedFS_Complexity import *

In [ ]:
import os
os.chdir("..")
root_path = os.getcwd()

In [ ]:
root_path

### Resultados distributed

In [ ]:
# Dataset 2
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset2_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset2")

In [ ]:
# Dataset  7
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset7_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset7")

In [ ]:
# Dataset 12
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset12_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset12")

In [ ]:
# # Dataset 14
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset14_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset14")

In [ ]:
# # Dataset 18
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset18_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset18")

In [ ]:
# # Dataset 20
df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset20_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)

plot_complexity_importances_by_model(df_random, dataset_name="Dataset20")

In [ ]:
# # Dataset 21
# df_random = pd.read_csv("Results_FS_Distributed_CV/ArtificialDataset21_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv", index_col=0)
#
# plot_complexity_importances_by_model(df_random, dataset_name="Dataset21")

VER CUÁLES SON SIEMPRE POSITIVAS Y ANALIZAR CÓMO VA LA PERFORMANCE SOLO CON ELLAS, sacar cuántas son prositivas, corr entre modelos del ranking y relacionar el número de positivas con el número de informativas

## Tabla de comparación SOTA vs distributed (backward)

In [ ]:
# Leer todos los comparison.csv de la carpeta
files = glob.glob("Results_ComparisonDistributed_SOTA/*_ComparisonTable_CV_neg_high_DatasetVersions.csv")

all_tables = []
for f in files:
    df = pd.read_csv(f, index_col=[1])
    all_tables.append(df)

comparison_all = pd.concat(all_tables)
comparison_all.head()

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset2'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset7'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset12'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset14'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset18'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset20'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset21'])
# En este todavía no se ha ejecutado

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset18a'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset18b'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset18c'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset20a'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset20b'])

In [ ]:
display(comparison_all.loc[comparison_all['dataset']=='ArtificialDataset20c'])

## Performance evolutiva

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset2_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset2_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset2",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset2",measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset7_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset7_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset7",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset7", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset12_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset12_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset12",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset12", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset14_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset14_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset14",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset14", measure="gps_test")

En este caso se aprecia que la performance comienza a disminuir aunque no den valores negativos.

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset18_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset18",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset18", measure="gps_test")

En el dataset 18 vemos que la mayor parte de los máximos se logran con variables que ya aumentan la complejidad (zona gris). Se trata de un conjunto de datos bastante complejo.

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18a_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset18a_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset18a",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset18a", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18b_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset18b_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset18b",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset18b", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18c_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset18c_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset18c",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset18c", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset20_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset20",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset20", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20a_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset20a_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset20a",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset20a", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20b_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset20b_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset20b",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset20b", measure="gps_test")

In [ ]:
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20c_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)
importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)
perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset20c_OutHigh_EvolutivePerformance.csv')

plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset20c",measure="acc_test")

In [ ]:
plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset20c", measure="gps_test")

La SVM se comporta como esperamos pero el knn no. Su performance sigue aumentando hasta el final. Más adelante estudiamos cómo se distribuye la tipología de datos. Tampoco entiendo muy bien la diferencia de comportamiento entre los 2 modelos.

In [ ]:
# path_csv = 'Results_FS_Distributed_CV/ArtificialDataset21_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
# importances_dict = load_importances_per_fold(path_csv)
# importances_all = pd.concat([v["kDN_importances_norm"] for v in importances_dict.values()],ignore_index=True)
#
# dfs = []
# for fold, v in importances_dict.items():
#     df = v["kDN_importances_norm"].copy()
#     df["fold"] = fold
#     dfs.append(df)
#
# importances_all = pd.concat(dfs, ignore_index=True)
# perf = pd.read_csv('Results_FS_Distributed_CV/ArtificialDataset21_OutHigh_EvolutivePerformance.csv')
#
# plot_incremental_performance(perf, importances_all,dataset="ArtificialDataset21",measure="acc_test")

In [ ]:
# plot_incremental_performance(perf, importances_all, dataset="ArtificialDataset21", measure="gps_test")

## Caracterización de variables

In [ ]:
dataset_name = 'ArtificialDataset2'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=1000,n_informative=10,n_noise=2,
                                         n_redundant_linear=4,n_redundant_nonlinear=2,
                                    flip_y=0, class_sep = 0.6, n_clusters_per_class=1 , weights=[0.5],
                                                     random_state=0,noise_std=0.01)
path_csv = 'Results_FS_Distributed_CV/ArtificialDataset2_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))


In [ ]:
df_folds

Pilla bien las informativas

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


In [ ]:
#### Dataset 7
dataset_name = 'ArtificialDataset7'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=1000,n_informative=20,n_noise=10,
                                         n_redundant_linear=10,n_redundant_nonlinear=10,
                                        flip_y=0, class_sep=1, n_clusters_per_class=1, weights=[0.5],
                                                     random_state=589,noise_std=0.05)

path_csv = 'Results_FS_Distributed_CV/ArtificialDataset7_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))

In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

Más o menos pilla bien las variables (el orden)

In [ ]:
#### Dataset 12
dataset_name = 'ArtificialDataset12'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=3000,n_informative=25,n_noise=30,
                                         n_redundant_linear=30,n_redundant_nonlinear=30,
                                        flip_y=0.2, class_sep=0.9, n_clusters_per_class=1, weights=[0.4],
                                                     random_state=987,noise_std=0.5)

path_csv = 'Results_FS_Distributed_CV/ArtificialDataset12_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))

In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

Se nos empiezan a colar más variablees redundants. No ruido, si no redundantes.

In [ ]:
#### Dataset 14
dataset_name = 'ArtificialDataset14'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=3000,n_informative=30,n_noise=40,
                                         n_redundant_linear=30,n_redundant_nonlinear=40,
                                        flip_y=0.2, class_sep=0.6, n_clusters_per_class=2, weights=[0.3],
                                                     random_state=95,noise_std=0.5)

path_csv = 'Results_FS_Distributed_CV/ArtificialDataset14_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))


In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

Se nos cuelan algunas redudantes (nada de ruido), pero generalmente bien.

In [ ]:
#### Dataset 18
dataset_name = 'ArtificialDataset18'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=500,n_informative=70,n_noise=40,
                                         n_redundant_linear=40,n_redundant_nonlinear=40,
                                        flip_y=0.4, class_sep=0.8, n_clusters_per_class=2, weights=[0.2],
                                                     random_state=9462,noise_std=0.5)

path_csv = 'Results_FS_Distributed_CV/ArtificialDataset18_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))

In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

El problema aquí es que se nos mete mucho ruido. Este dataset es difícil y es en el que más ruido se nos cuela.

In [ ]:
#### Dataset 20
dataset_name = 'ArtificialDataset20'
X, y, dict_info_feature = generate_synthetic_dataset(n_samples=500,n_informative=300,n_noise=60,
                                         n_redundant_linear=60,n_redundant_nonlinear=60,
                                        flip_y=0.1, class_sep=0.6, n_clusters_per_class=1, weights=[0.3],
                                                     random_state=4556,noise_std=0.5)


path_csv = 'Results_FS_Distributed_CV/ArtificialDataset20_DistributedCVRandom_OutHigh_FeatureImportance_Folds.csv'
importances_dict = load_importances_per_fold(path_csv)

dfs = []
for fold, v in importances_dict.items():
    df = v["kDN_importances_norm"].copy()
    df["fold"] = fold
    dfs.append(df)

importances_all = pd.concat(dfs, ignore_index=True)


ranking = importances_all.loc[importances_all.fold==1]
dep2 = build_dependencies(dict_info_feature)
type_dict2 = build_type_dict(dict_info_feature)
results = caracterize_features_ranking(ranking['feature'], type_dict2, dep2)


summary, df_folds, labels = analyze_topk_with_labels(
    df_rankings=importances_all,
    type_dict=type_dict2,
    dependencies=dep2,
    k_real=len(dict_info_feature['informative']))

In [ ]:
df_folds

In [ ]:
df_positions, pos_by_type, labels = analyze_position_distribution(df_rankings=importances_all,
    type_dict=type_dict2,dependencies=dep2)


plt.figure(figsize=(10,5))
sns.boxplot(data=df_positions, x="label", y="pos", width=0.2, boxprops={'facecolor':'none'})
plt.title("Distribución de posiciones por tipo de variable")
plt.ylabel("Posición en el ranking (menor = mejor)")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

Este es menos complejo y se cuela menos ruido, pero también se mete una cantidad importante.

PRÓXIMOS PASOS:
- Probar con distintos niveles de dificultad con la misma cantidad de variables para ver dónde está el problema.
- También poniendo distinta cantidad de variables informativas porque creo que poner 300 no es real.
- Hacer mini prueba con datos reales por si los estoy generando de forma rara.
- Ejecutar varias veces el conteo de variables para ver que es estable
- Verificar cómo se comportan métodos del SOTA en los datasets más complejos, quizás simplemente todo va mal ahí por las características de los datos y no tiene sentido intentar mejorar lo imposible.
- Recomendaciones de generación de datos para ML, más ceentrado en FS.